In [ ]:
import sys
from pyprojroot import here

# Ritorna il percorso assoluto della root del progetto
PROJECT_ROOT = str(here()) + "/"
sys.path.append(PROJECT_ROOT)
sys.dont_write_bytecode = True

print(f"La root del progetto è: {PROJECT_ROOT}")

In [ ]:
from paths import DRT_PATH, TP3_PATH, MODEL_PATH, ROCK_PATH

ABS_PATH = PROJECT_ROOT + DRT_PATH + TP3_PATH
LOAD_MODEL = PROJECT_ROOT + MODEL_PATH + ROCK_PATH

model_name = "Rock1.blend"

In [ ]:
# Create needed folders
import os

lst_folders = ["figures", "files", "models"]

for name in lst_folders:
    os.makedirs(os.path.join(ABS_PATH, name), exist_ok=True)

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from modelaquisition.bl2pina import Blend2Pina
from modelaquisition.bl2msh import Blend2Mesh
from modelaquisition.msh2xdmf import Msh2Xdmf

In [ ]:
torch.set_default_dtype(torch.float64)

# Simulation of weather data

In [ ]:
seed = 10
np.random.seed(seed)

In [ ]:
number_samples = 100
mean_temperature = 22.4
std_temperature = 1.2

temperature = np.random.normal(
    loc=mean_temperature,
    scale=std_temperature,
    size=(number_samples)
)

Removing outliers and substituting with estimation

In [ ]:
over_std_idx = np.logical_and(abs(temperature - mean_temperature) - 2*std_temperature >= 0, temperature - mean_temperature >= 0)
under_std_idx = np.logical_and(abs(temperature - mean_temperature) - 2*std_temperature >= 0, temperature - mean_temperature <= 0)
temperature[over_std_idx] = mean_temperature + 2*std_temperature
temperature[under_std_idx] = mean_temperature - 2*std_temperature

In [ ]:
half = int(number_samples/2)

first_part = temperature[:half]
first_part.sort()
second_part = temperature[half:]
second_part.sort()
second_part = np.flip(second_part)

temperature = np.concatenate([first_part, second_part])

In [ ]:
date_range = pd.date_range(start='2024-01-01 00:00:00', end='2024-01-01 23:59:00', periods=number_samples)
temperature_data = pd.DataFrame({'Datetime': date_range, 'Temperature_C': temperature})

In [ ]:
mean_plus_std = mean_temperature + std_temperature
mean_minus_std = mean_temperature - std_temperature

plt.figure(figsize=(20, 6))
plt.scatter(temperature_data['Datetime'], temperature_data['Temperature_C'], color='blue', marker="p")
plt.axhline(mean_temperature, color='red', linestyle='--', label='Mean', linewidth=2.2)
plt.axhline(mean_plus_std, color='orange', linestyle='-.', label='Mean + Std Dev', linewidth=2.2)
plt.axhline(mean_minus_std, color='orange', linestyle=':', label='Mean - Std Dev', linewidth=2.2)
plt.legend()
plt.title("Sampled data related to surface temperature")
plt.xlabel("time")
plt.ylabel("Temperature  (°C)")
plt.grid(True)
plt.xticks(temperature_data['Datetime'], temperature_data['Datetime'].dt.strftime('%H:%M'), rotation=45)
plt.tight_layout()
plt.savefig(ABS_PATH+"figures/weather_data.png", transparent=True, dpi=300)
plt.show()

In [ ]:
temperature_data.to_csv(ABS_PATH+"files/tempdata.csv", sep=";", index=False)

Connect temperature data with boundary points

In [ ]:
num_points_b = 100

rock = Blend2Pina(LOAD_MODEL + model_name)
surface = rock.boundary()
points_boundary = surface.sample(num_points_b)

LT_lst = list()
time = np.linspace(0, 1, len(temperature_data))
T_boundary = list()

for i in range(1, time.shape[0]):
  for sp_el in points_boundary:
    LT_lst.append([sp_el.tensor[0].item(), sp_el.tensor[1].item(), sp_el.tensor[2].item(), time[i], temperature_data["Temperature_C"][i]])

data_boundary = pd.DataFrame(LT_lst, columns=["x", "y", "z", "t", "u"])
data_boundary.to_csv(ABS_PATH+"files/data.csv", sep=";", index=None)

# Sistema di PDE in considerazione

Consideriamo il problema sul dominio $\Omega$ di frontiera $\Gamma = \partial \Omega$
$$
\begin{equation}
    \begin{cases}
        \Delta u = & \left(
                \begin{matrix}
                    2 u_1\\
                    2
                \end{matrix}
            \right), & \left(x,y,z\right) \in \Omega \\
        u \left( x, y, z \right) = & \left(
                \begin{matrix}
                    e^{x+y}\\
                    x^2 - z
                \end{matrix} \right) & \left( x, y, z \right) \in \Gamma
    \end{cases}
    \tag{1}
\end{equation}
$$

di soluzione analitica $u : \Omega \subseteq \mathbb{R}^3 \longmapsto \mathbb{R}^2$
$$
\begin{equation}
    u \left( x, y, z \right) = \left(
                \begin{matrix}
                    e^{x+y}\\
                    x^2 - z
                \end{matrix} \right) \qquad \left( x, y, z \right) \in \Omega
    \tag{2}
\end{equation}
$$

## Creazione dei dati al contorno

In [ ]:
num_points_int = 10_000
domain = rock.intern()
points_internal = domain.sample(num_points_int)

In [ ]:
df_int = pd.DataFrame(
    points_internal.tensor.detach().numpy()
)
df_int.to_csv("./files/input_int.csv", sep = ";")

## Creazione mesh e xdmf

In [ ]:
rock_msh = Blend2Mesh(LOAD_MODEL + model_name, "rock")

In [ ]:
rock_msh.create_mesh(len_msh=0.07)

In [ ]:
rock_xdmf = Msh2Xdmf("rock.msh", "rock")
rock_xdmf.to_xdmf()